# Data Partitioning Report

**Input:** cleaned Jigsaw toxicity data  
**Partition design:** 70% training, 15% validation, 15% test

## Summary

The cleaned dataset is divided at the exact-text-group level, keeping repeated comments in a single partition. A fixed random seed provides a reproducible allocation and supporting audit records.


In [1]:
from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
SPLIT_NAMES = ("train", "validation", "test")
FRACTIONS = np.array([0.70, 0.15, 0.15])
OVERWRITE = False  # Only set True if you intentionally want to regenerate saved splits.

if len(FRACTIONS) != 3 or np.any(FRACTIONS <= 0) or not np.isclose(FRACTIONS.sum(), 1):
    raise ValueError("Choose three positive split fractions that sum to 1.")


## Input validation

The partitioning data include comment IDs, text, fractional targets, binary targets, and exact-text group identifiers. Checks confirm field completeness, unique IDs, valid labels, and internally consistent groups.


In [2]:
candidates = [
    Path("data/processed/cleaned_train.csv.gz"),
    Path("../data/processed/cleaned_train.csv.gz")
]
data_path = next((path.resolve() for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        "Cannot find data/processed/cleaned_train.csv.gz. "
        "Run the cleaning notebook's save cell first."
    )

df = pd.read_csv(data_path)
required = {"id", "comment_text", "target", "toxic", "text_group"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns from the cleaning notebook: {missing}")
if df.empty or df[list(required)].isna().any().any():
    raise ValueError("The cleaned data is empty or has missing required values.")
if df["id"].duplicated().any():
    raise ValueError("Comment IDs must be unique.")
if not df["target"].between(0, 1).all():
    raise ValueError("Toxicity targets must range from 0 to 1.")
if not df["toxic"].isin([0, 1]).all():
    raise ValueError("The toxic column must contain only 0 and 1.")
if not df["toxic"].eq(df["target"].ge(0.5).astype(int)).all():
    raise ValueError("The toxic column does not match the target >= 0.5 convention.")
if not df["comment_text"].str.strip().ne("").all():
    raise ValueError("Blank comments remain; rerun the cleaning notebook.")

print(f"Loaded {len(df):,} comments with {df.shape[1]} columns.")
print(f"Overall toxic share: {100 * df['toxic'].mean():.2f}%")


Loaded 1,804,871 comments with 47 columns.
Overall toxic share: 8.00%


## Method

Exact-text groups are classified as non-toxic only, toxic only, or mixed-label. Groups are shuffled within each category using seed 42 and distributed by largest-remainder allocation. This maintains approximate class composition while preventing exact-text leakage.

Group sizes vary, so row percentages and toxicity rates are approximate. Identity variables are not used for allocation, and near-duplicates or related discussions may still occur across partitions.


In [3]:
# Each unique text must have exactly one group, and each group exactly one text.
text_mapping = df[["text_group", "comment_text"]].drop_duplicates()
if text_mapping["text_group"].duplicated().any() or text_mapping["comment_text"].duplicated().any():
    raise ValueError("text_group does not consistently identify exact comment text.")
del text_mapping

group_table = df.groupby("text_group", sort=True).agg(
    rows=("id", "size"),
    min_toxic=("toxic", "min"),
    max_toxic=("toxic", "max")
)
group_table["stratum"] = np.select(
    [group_table["max_toxic"].eq(0), group_table["min_toxic"].eq(1)],
    ["non_toxic_only", "toxic_only"],
    default="mixed"
)
if len(group_table) < 3:
    raise ValueError("At least three unique text groups are required.")

def allocate_text_groups(groups, fractions, seed):
    """Assign whole text groups, shuffling separately within each label stratum."""
    rng = np.random.default_rng(seed)
    assigned = pd.Series(index=groups.index, dtype="object", name="split")
    for stratum in sorted(groups["stratum"].unique()):
        # Sorting group IDs makes this independent of dataframe row order.
        group_ids = groups.index[groups["stratum"].eq(stratum)].sort_values().to_numpy()
        shuffled = rng.permutation(group_ids)
        desired = len(shuffled) * fractions
        counts = np.floor(desired).astype(int)
        remainder_order = np.argsort(-(desired - counts), kind="stable")
        counts[remainder_order[:len(shuffled) - counts.sum()]] += 1
        cursor = 0
        for split_name, count in zip(SPLIT_NAMES, counts):
            assigned.loc[shuffled[cursor:cursor + count]] = split_name
            cursor += count
    if assigned.isna().any():
        raise AssertionError("Some text groups were not assigned.")
    return assigned

print(f"Unique text groups: {len(group_table):,}")
display(group_table.groupby("stratum").agg(text_groups=("rows", "size"), comments=("rows", "sum")))


Unique text groups: 1,780,822


,text_groups,comments
stratum,,
mixed,596,2077
non_toxic_only,1638063,1659408
toxic_only,142163,143386


## Partition verification

Each comment inherits the assignment of its text group. Verification covers complete row coverage, unique IDs, no exact-text overlap, representation of both classes, and reproducibility under the fixed seed.


In [4]:
group_assignment = allocate_text_groups(group_table, FRACTIONS, SEED)
split_labels = df["text_group"].map(group_assignment)
if split_labels.isna().any():
    raise AssertionError("Some comments have no assigned split.")
if set(split_labels.unique()) != set(SPLIT_NAMES):
    raise ValueError("At least one split is empty. More text groups are needed.")

assignment = df[["id", "text_group"]].copy()
assignment["split"] = split_labels

assert len(assignment) == len(df)
assert assignment["id"].is_unique
assert assignment.groupby("text_group")["split"].nunique().eq(1).all()
assert group_assignment.equals(allocate_text_groups(group_table, FRACTIONS, SEED))

summary_rows = []
for name, fraction in zip(SPLIT_NAMES, FRACTIONS):
    mask = split_labels.eq(name)
    toxic_count = int(df.loc[mask, "toxic"].sum())
    count = int(mask.sum())
    summary_rows.append({
        "split": name,
        "comments": count,
        "actual_percent": round(100 * count / len(df), 2),
        "requested_percent": round(100 * fraction, 2),
        "text_groups": int(group_assignment.eq(name).sum()),
        "non_toxic_comments": count - toxic_count,
        "toxic_comments": toxic_count,
        "toxic_percent": round(100 * toxic_count / count, 2)
    })
split_summary = pd.DataFrame(summary_rows)
display(split_summary)

if not df.groupby(split_labels)["toxic"].nunique().eq(2).all():
    raise ValueError("At least one split lacks a toxicity class. Inspect the dataset before proceeding.")
print("Passed: full coverage, unique IDs, no exact-text overlap, and reproducible assignment.")


,split,comments,actual_percent,requested_percent,text_groups,non_toxic_comments,toxic_comments,toxic_percent
0,train,1263592,70.01,70.0,1246575,1162583,101009,7.99
1,validation,270541,14.99,15.0,267125,248859,21682,8.01
2,test,270738,15.00,15.0,267122,249095,21643,7.99


Passed: full coverage, unique IDs, no exact-text overlap, and reproducible assignment.


## Identity coverage

Identity mentions use a threshold of 0.5, while missing annotations remain excluded. Counts summarize partition composition and may overlap across groups. Fewer than 100 non-toxic mentions is flagged as a limited-sample basis for subgroup false-positive-rate estimates.


In [5]:
identity_columns = [
    "male", "female", "transgender", "other_gender", "heterosexual",
    "homosexual_gay_or_lesbian", "bisexual", "other_sexual_orientation",
    "christian", "jewish", "muslim", "hindu", "buddhist", "atheist",
    "other_religion", "black", "white", "asian", "latino",
    "other_race_or_ethnicity", "physical_disability",
    "intellectual_or_learning_disability", "psychiatric_or_mental_illness",
    "other_disability"
]
identity_columns = [column for column in identity_columns if column in df.columns]
identity_rows = []
for column in identity_columns:
    for name in SPLIT_NAMES:
        in_split = split_labels.eq(name)
        mentioned = in_split & df[column].ge(0.5)
        non_toxic = int((mentioned & df["toxic"].eq(0)).sum())
        identity_rows.append({
            "identity": column,
            "split": name,
            "annotated_comments": int((in_split & df[column].notna()).sum()),
            "identity_mentions": int(mentioned.sum()),
            "non_toxic_mentions": non_toxic,
            "toxic_mentions": int((mentioned & df["toxic"].eq(1)).sum()),
            "fewer_than_100_non_toxic_mentions": non_toxic < 100
        })
identity_summary = pd.DataFrame(identity_rows, columns=[
    "identity", "split", "annotated_comments", "identity_mentions",
    "non_toxic_mentions", "toxic_mentions", "fewer_than_100_non_toxic_mentions"
])
if identity_columns:
    coverage = identity_summary.pivot(index="identity", columns="split", values="non_toxic_mentions")
    display(coverage.reindex(columns=SPLIT_NAMES))
    limited = identity_summary.loc[
        identity_summary["split"].isin(["validation", "test"])
        & identity_summary["fewer_than_100_non_toxic_mentions"]
    ]
    print(f"Validation/test identity entries with fewer than 100 non-toxic mentions: {len(limited)}")
else:
    warnings.warn("No identity columns found. Fairness evaluation requires identity annotations.")


split,train,validation,test
identity,,,
asian,2879,570,563
atheist,853,181,193
bisexual,151,43,31
black,7241,1496,1486
buddhist,375,72,72
christian,25812,5437,5501
female,32247,7014,6857
heterosexual,698,156,143
hindu,359,85,75


Validation/test identity entries with fewer than 100 non-toxic mentions: 20


## Saved artifacts

| File | Contents |
| --- | --- |
| `train.csv.gz`, `validation.csv.gz`, `test.csv.gz` | fixed model-development partitions |
| `split_assignments.csv.gz` | ID-level partition membership |
| `split_summary.csv` | partition size and class composition |
| `identity_split_summary.csv` | identity coverage by partition |
| `split_metadata.json` | seed, fractions, input checksum, and allocation details |


In [6]:
output_dir = data_path.parent.parent / "splits"
output_dir.mkdir(parents=True, exist_ok=True)
output_names = [f"{name}.csv.gz" for name in SPLIT_NAMES] + [
    "split_assignments.csv.gz", "split_summary.csv",
    "identity_split_summary.csv", "split_metadata.json"
]
existing = [name for name in output_names if (output_dir / name).exists()]
if existing and not OVERWRITE:
    raise FileExistsError(
        f"Split outputs already exist: {existing}. Keep these shared splits, "
        "or intentionally enable OVERWRITE in the first cell."
    )

hasher = hashlib.sha256()
with data_path.open("rb") as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b""):
        hasher.update(chunk)

for name in SPLIT_NAMES:
    destination = output_dir / f"{name}.csv.gz"
    df.loc[split_labels.eq(name)].to_csv(destination, index=False, compression="gzip")
    print(f"Saved {name}: {destination}")

assignment.to_csv(output_dir / "split_assignments.csv.gz", index=False, compression="gzip")
split_summary.to_csv(output_dir / "split_summary.csv", index=False)
identity_summary.to_csv(output_dir / "identity_split_summary.csv", index=False)
metadata = {
    "input_filename": data_path.name,
    "input_sha256": hasher.hexdigest(),
    "seed": SEED,
    "requested_fractions": dict(zip(SPLIT_NAMES, FRACTIONS.tolist())),
    "method": "Seeded random allocation within non-toxic-only, toxic-only, and mixed exact-text groups; largest-remainder group counts",
    "identity_stratified": False,
    "rows": len(df),
    "text_groups": len(group_table),
    "binary_target_threshold": 0.5,
    "identity_mention_threshold": 0.5,
    "split_summary": split_summary.to_dict(orient="records"),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__
}
(output_dir / "split_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print("Saved assignments, summaries, and split metadata.")


Saved train: /Users/sylviadu/Documents/DATA 4620/Bert-Toxicity-Identification/data/splits/train.csv.gz
Saved validation: /Users/sylviadu/Documents/DATA 4620/Bert-Toxicity-Identification/data/splits/validation.csv.gz
Saved test: /Users/sylviadu/Documents/DATA 4620/Bert-Toxicity-Identification/data/splits/test.csv.gz
Saved assignments, summaries, and split metadata.


## Evaluation protocol

Training uses the training partition, model selection uses validation data, and the test partition is reserved for final comparison. Identity annotations may support weighting and subgroup analysis, but the BERT input consists only of comment text.
